# Flux.2 [klein] 9B Edit Web Server (ComfyUI) — Google colab

ComfyUI APIをバックエンドとして、Web サーバーを起動し、cloudflared で外部公開します。

**必要環境:**
 - GPUが使えるランタイム(A100)

## 1. torchのインストール

In [ ]:
!pip install -U huggingface_hub requests urllib3 charset-normalizer

import os
import sys
import subprocess
import platform

def get_system_cuda_version():
    """nvidia-smiを使用してシステムのCUDAバージョンを取得する"""
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits"]).decode()
        # CUDAバージョンを直接取得する（もしくはnvccから取得）
        cuda_out = subprocess.check_output(["nvcc", "--version"]).decode()
        for line in cuda_out.split('\n'):
            if "release" in line:
                return line.split("release ")[1].split(",")[0].replace(".", "")
    except:
        # 万が一取得失敗した場合はデフォルトを指定（例: 130）
        return "130"
    return "130"

def run_command(command):
    print(f"Executing: {command}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + command.split())

# 1. CUDAバージョンの取得
cuda_tag = f"cu{get_system_cuda_version()}"
print(f"🚀 Detected System CUDA: {cuda_tag}")

# 2. PyTorchのインストール (CUDAバージョンに合わせる)
print(f"📦 Installing PyTorch for {cuda_tag}...")
torch_index_url = f"https://download.pytorch.org/whl/{cuda_tag}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch", "torchvision", "--index-url", torch_index_url])

print("\nInstallations completed successfully!")

## llama-cpp-pythonのインストール

In [ ]:
import sys
import platform
import requests
import torch
import os

def get_compatible_wheel_url(repo="JamePeng/llama-cpp-python", search_limit=15):
    # --- 1. 詳細な環境情報の取得 ---
    # sys.version_infoを直接数値として扱う
    major = sys.version_info.major
    minor = sys.version_info.minor
    py_ver = f"cp{major}{minor}"
    
    # 実行中のPythonのフルパスとバージョンをログ出力（デバッグ用）
    print(f"--- Environment Debug Info ---")
    print(f"Python Executable: {sys.executable}")
    print(f"Python Version: {sys.version}")
    
    # プラットフォーム判定の厳密化
    is_win = platform.system().lower() == "windows"
    if is_win:
        plat = "win_amd64"
    else:
        # Linux系 (manylinuxなどの互換性タグが含まれる場合を考慮)
        plat = "linux_x86_64"

    # CUDAタグの生成 (13.0 -> cu130)
    cuda_tag = ""
    if torch.cuda.is_available() and torch.version.cuda:
        # ドットを除去 (13.0 -> 130)
        cuda_val = torch.version.cuda.split('+')[0] # 13.0.xxx 対応
        cuda_clean = cuda_val.replace('.', '')
        # 万が一 13.0 が 130 ではなく 13 になるのを防ぐ（2桁の場合は後ろに0）
        if len(cuda_clean) == 2:
            cuda_clean += "0"
        cuda_tag = f"cu{cuda_clean}"
    else:
        cuda_tag = "cpu"

    print(f"🔍 判定スコア: Python={py_ver}, CUDA={cuda_tag}, Platform={plat}")
    print(f"📡 {repo} から最適なバイナリを探索中...")

    # --- 2. GitHub APIでリリースのリストを取得 ---
    try:
        api_url = f"https://api.github.com/repos/{repo}/releases"
        # ページネーションを考慮して少し多めに取得
        response = requests.get(api_url, params={"per_page": search_limit}, timeout=10)
        response.raise_for_status()
        releases = response.json()
    except Exception as e:
        print(f"❌ APIエラー: {e}")
        return None

    # --- 3. マッチング処理 ---
    for release in releases:
        assets = release.get("assets", [])
        for asset in assets:
            name = asset["name"].lower() # 小文字で統一して比較
            if not name.endswith(".whl"):
                continue
            
            # 条件判定: Pythonバージョン、プラットフォーム、CUDAタグ
            # JamePeng版の命名例: llama_cpp_python-0.3.7+cu130-cp310-cp310-win_amd64.whl
            if (py_ver in name) and (plat in name) and (cuda_tag in name):
                return asset["browser_download_url"]

    return None

# --- 実行 ---
wheel_url = get_compatible_wheel_url()

if wheel_url:
    print(f"\n✅ 最適なバイナリが見つかりました:\n{wheel_url}")
    !pip install -U "{wheel_url}"
else:
    print("\n❌ 条件に一致するバイナリが見つかりませんでした。")


## 2. パッケージインストール

In [ ]:
# ライブラリインストール
!apt -y remove python3-blinker
!pip install -U "flask>=3.0"  pillow huggingface_hub psutil accelerate googletrans huggingface_hub requests urllib3 charset-normalizer soundfile gguf --extra-index-url $torch_index_url

# cloudflared インストール（）
!wget -qN https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!git pull

%mkdir -p models/LLM

# 追加インストール
!pip install -r requirements.txt --extra-index-url $torch_index_url

%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
!git clone https://github.com/id-fa/ComfyUI-QwenVL.git
    
%cd ../../

print("OK")

## 3. HuggingFace ログイン

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("HuggingFace: Colab Secrets で認証し、環境変数を設定しました")
    else:
        raise Exception("HF_TOKEN is empty")

except Exception:
    print("Colab Secrets に HF_TOKEN が未登録です。手動ログインを試みます...")
    from huggingface_hub import login
    login()


## 4. モデルのダウンロード
FLUX.2-klein-9b-fp8は一度ブラウザでアクセスして承認リクエストする必要があります。

https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-fp8

認証済でエラーが出る場合はトークンの権限編集で「Read access to contents of all public gated repos you can access」をチェック

In [ ]:
# transformer 要ログイン＆認証 
!hf download black-forest-labs/FLUX.2-klein-9b-fp8 --local-dir ./hf_download --include "flux-2-klein-9b-fp8.safetensors"
%mv ./hf_download/flux-2-klein-9b-fp8.safetensors ./ComfyUI/models/diffusion_models/

# text encoder
!hf download Comfy-Org/flux2-klein-9B --local-dir ./hf_download --include "split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors"
%mv ./hf_download/split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors ./ComfyUI/models/text_encoders/

# vae
!hf download black-forest-labs/FLUX.2-small-decoder --local-dir ./hf_download --include "full_encoder_small_decoder.safetensors"
%mv ./hf_download/full_encoder_small_decoder.safetensors ./ComfyUI/models/vae/

# prompt enhancer
!hf download jwest33/qwen3.5-9b-null-space-abliterated-GGUF --local-dir ./hf_download/jwest33/qwen3.5-9b-null-space-abliterated-GGUF --include "qwen3.5-9b-null-space-abliterated-q8_0.gguf"
!hf download jwest33/qwen3.5-9b-null-space-abliterated-GGUF --local-dir ./hf_download/jwest33/qwen3.5-9b-null-space-abliterated-GGUF --include "mmproj-qwen3.5-9b-f16.gguf"
%mv ./hf_download/jwest33 ./ComfyUI/models/LLM/

# prompt enhancer gemma-4 test
#!hf download mradermacher/gemma-4-E4B-it-heretic-ara-GGUF --local-dir ./hf_download/mradermacher/gemma-4-E4B-it-heretic-ara-GGUF --include "gemma-4-E4B-it-heretic-ara.Q8_0.gguf"
#!hf download mradermacher/gemma-4-E4B-it-heretic-ara-GGUF --local-dir ./hf_download/mradermacher/gemma-4-E4B-it-heretic-ara-GGUF --include "gemma-4-E4B-it-heretic-ara.mmproj-Q8_0.gguf"
#%mv ./hf_download/mradermacher ./ComfyUI/models/LLM/

print("OK")

## 5. app_comfyui.py を配置

In [ ]:
import shutil

!git clone https://github.com/id-fa/simple-image-edit-with-qwen.git repo_tmp
!cp repo_tmp/server/app_comfyui_klein.py .
!cp -r repo_tmp/server/lib .
!cp -r repo_tmp/server/comfyui_workflow .

# gemma-4 test
#!cp -f ./comfyui_workflow/gemma4_test/enhance_prompt_api.json ./comfyui_workflow/

print("準備完了\n")
!ls -la app_* comfyui_workflow 2>/dev/null || echo "⚠ ファイルが見つかりません。"

## 6. LoRAのダウンロード（sample）

In [ ]:
%mkdir ./LoRA

!hf download diroverflo/FLux_Klein_9B_NSFW --local-dir ./LoRA/ --include "Flux Klein - NSFW v2.safetensors"
!hf download lrzjason/Anything2Real --local-dir ./LoRA/ --include "f2k_anything2real_a.safetensors"
!hf download Alissonerdx/BFS-Best-Face-Swap --local-dir ./LoRA/ --include "bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors"

print("OK")

## 7. 設定

パスワードやプリセットを変更できます。

In [ ]:
# --- 設定 ---
PASSWORD = "password"       # 生成パスワード
PORT = 18080                 # サーバーポート
GALLERY = True              # ギャラリーモード有効化

# プリセット (空リストならボタン非表示)
PRESETS = [
    "高画質化::Improve image quality and sharpness.",
    "テキスト除去::Remove all overlaid text, subtitles, and credits.",
    "画像2の服を着せる::将图像 1 中的角色穿上图像 2 中的服装。无视物理定律，穿一模一样的服装。不要添加额外的绳子或布料。",
    "画像2のポーズにする::The character in image 1 is in the pose of image 2",
#    "ポロリ(NSFW)::只脱掉胸部的衣服",
    "LoRA-Anything2Real::change the picture 1 to realistic photograph, ",
    "LoRA-BFS顔交換::replace the head, face and hair.",
]

COMFYUI_LOG = "./comfyui.log"
SERVER_LOG = "./server.log"

print("OK")

## 8. ComfyUI起動

In [ ]:
import subprocess, time, re, threading

# ログファイルに出力（パイプバッファ詰まり防止）
clog_fh = open(COMFYUI_LOG, "w")

# ComfyUI起動
# --use-flash-attention は使える環境のみ有効にする
comfyui_proc = subprocess.Popen(
    [
        "python", "-u", "./ComfyUI/main.py",
        "--listen",
        "--port", "6006",
        "--preview-method", "auto",
        "--dont-upcast-attention",
#       "--use-flash-attention",
    ],
    stdout=clog_fh, stderr=clog_fh, text=True
)

print("OK")

## 9. サーバー起動

In [ ]:
# コマンドライン引数を構築
cmd = [
    "python", "-u", "./app_comfyui_klein.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--comfyui-url", "http://127.0.0.1:6006",
    "--comfyui-path", "./ComfyUI",
    "--password", PASSWORD,
    "--steps", "8",
]
if GALLERY:
    cmd.append("--gallery")
for preset in PRESETS:
    cmd.extend(["--preset", preset])

print(f"starting server: {' '.join(cmd)}")

# ログファイルに出力（パイプバッファ詰まり防止）
log_fh = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

# ログを監視してサーバー起動完了を待機
ready = False
deadline = time.time() + 600  # 10分タイムアウト
seen = 0
while time.time() < deadline:
    time.sleep(2)
    if server_proc.poll() is not None:
        print("ERROR: server process exited unexpectedly")
        with open(SERVER_LOG) as f:
            print(f.read())
        break
    with open(SERVER_LOG) as f:
        lines = f.readlines()
    # 新しい行を表示
    for line in lines[seen:]:
        print(line, end="")
    seen = len(lines)
    if any("server starting at" in l for l in lines):
        ready = True
        break

if not ready:
    raise RuntimeError("Server failed to start. Check server.log")

print(f"\n[server is running on port {PORT}")

## 10. cloudflared 公開

cloudflared で公開URLを生成します。
出力に表示される `https://******.trycloudflare.com` のURLにアクセスしてください。

In [ ]:
print(f"starting cloudflared tunnel...")

# cloudflared 起動 (stderr にURL出力)
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

# stderr を読み続けるスレッド（バッファ詰まり防止）
tunnel_lines = []
def drain_tunnel_stderr():
    for line in iter(tunnel_proc.stderr.readline, ""):
        tunnel_lines.append(line)
threading.Thread(target=drain_tunnel_stderr, daemon=True).start()

# URL抽出を待機
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    time.sleep(1)
    for line in tunnel_lines:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            public_url = m.group(1)
            break
    if public_url:
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  Public URL: {public_url}")
    print(f"  Password:   {PASSWORD}")
    print(f"{'='*60}")
    print("30秒ほど待ってからアクセスしてください")
else:
    print("ERROR: cloudflared URL を取得できませんでした")
    print("tunnel log:", ''.join(tunnel_lines[-10:]))

print("バックグラウンドで実行中")

## 11. ユーティリティ

ログ確認・サーバー停止用。

In [ ]:
# サーバーログ確認（直近20行）
!tail -20 ./server.log
print("\n")

# comfyuiログ確認（直近20行）
!tail -20 ./comfyui.log

# サーバー + トンネル停止
```
server_proc.terminate()
log_fh.close()
print("stopped server")

comfyui_proc.terminate()
clog_fh.close()
print("stopped comfyui")

tunnel_proc.terminate()
print("stopped")
```

In [ ]:
# コピペ実行用
